# 탐색적 데이터 분석 (EDA)

In [ ]:
# 라이브러리 Import
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("="*60)

In [ ]:
# 1) 데이터 로드
product_type_1 = pd.read_csv('../data_processed/product_type_1.csv',header=[0,1])

print("="*60)
print("product_type_1 데이터 로드 완료!")
print("="*60)

In [ ]:
# 결측치 확인
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': product_type_1.isnull().sum(),
    '결측비율(%)': (product_type_1.isnull().sum() / len(product_type_1) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

In [ ]:
process_data = product_type_1['process']
sensor_data = product_type_1['sensor']

# X값 준비 - process, sensor 컬럼명 평탄화
X = pd.concat([process_data, sensor_data], axis=1)
X.columns = [f"process_{col}" for col in process_data.columns] + \
            [f"sensor_{col}" for col in sensor_data.columns]

# y 값 준비
defects_data = product_type_1['defects']

# 컬럼 범주화
defect_groups = {
    "표면": ["dent_1", "stain_1", "exfoliation_1", "exfoliation_2"],
    "구조": ["short_shot_1", "short_shot_2", "bubble_1", "bubble_2", "deformation_1", "deformation_2"],
}

# 불량유형 범주화 (그룹 내 하나라도 불량이면 1)
y = pd.DataFrame(index=defects_data.index)
for group, cols in defect_groups.items():
    usable = [c for c in cols if c in defects_data.columns]
    y[group] = defects_data[usable].max(axis=1) if usable else 0

print("="*60)
print(y.value_counts().sort_index())
print("="*60)

# 'stratify' 분할 기준용 생성
strata = y.astype(str).agg("".join, axis=1)  # 분할 기준용
print("stratify 분할 기준 데이터:")
print("="*60)
print(strata.value_counts().sort_index())

print("\n[표면/구조 불량 비율(%)]")
display((y.mean() * 100).round(2).to_frame("rate_%"))

In [ ]:
# train, test 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("="*60)
print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("="*60)
print("\n[Train set 타겟 분포]")
print("="*60)
print(y_train.value_counts())

print("="*60)
print("\n[Test set 타겟 분포]")
print("="*60)
print(y_test.value_counts())

In [ ]:
print(X_train.info())
print("=" * 60)
print(y_train.info())

In [ ]:
plt.subplots(figsize=(30,30))
sns.heatmap(data=X_train.corr(),linewidth=0.1,annot=True,fmt='.2f',cmap='coolwarm')